# Built-in fiber generators

TANGLE provides small crossing generators for mechanics tests and a
configurable population generator for material recipes. Generation is
seeded and deterministic; relaxation is tolerance-deterministic rather
than promised bitwise-identical across parallel backends.

In [ ]:
import tangle

cell = tangle.Cell([1e-3, 1e-3, 1e-3])
# Point crossings are the minimal rigid-contact demonstration.
point_crossing = tangle.generate_point_crossing(
    cell, count=8, length=0.8e-3, radius=9.5e-6,
    material_name="large", name="center crossing",
)
# Distinct placed/rest shape controls create initially bent fibers.
curved_crossing = tangle.generate_multisegment_crossing(
    cell, count=4, segments_per_fiber=8, length=0.8e-3,
    placed_chord_fraction=0.8, radius=9.5e-6,
    rest_shape="straight", rest_amplitude=0.0,
    placed_shape="curved", placed_amplitude=0.1e-3,
    minimum_bend_radius=50e-6,
)
# A two-fiber pair is useful for isolated refinement/contact tests.
pair = tangle.generate_fiber_pair_crossing(
    cell, segments_per_fiber=1, length=0.8e-3,
    radius=9.5e-6, axis_separation=10e-6,
    crossing_angle_degrees=90.0,
)
[len(point_crossing), len(curved_crossing), len(pair)]

## Every `FiberPopulationSettings` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `count` | Number of fibers to generate. | count |
| `segments_per_fiber` | Initial uniform centerline resolution. | count |
| `seed` | Deterministic sampling seed. | integer |
| `length_minimum` | Shortest sampled fiber. | m |
| `length_maximum` | Longest sampled fiber. | m |
| `nominal_parent_length` | Optional physical parent length for periodic fragments. | m or `None` |
| `radius_minimum` | Smallest sampled radius. | m |
| `radius_maximum` | Largest sampled radius. | m |
| `curvature_amplitude_minimum` | Minimum generated waviness amplitude. | m |
| `curvature_amplitude_maximum` | Maximum generated waviness amplitude. | m |
| `orientation` | Orientation distribution. | `isotropic_3d`, `planar`, `layered_biaxial`, `aligned` |
| `orientation_axis` | Normal or preferred direction used by the orientation model. | unit-like xyz vector |
| `maximum_angle` | Angular support around an aligned direction. | rad |
| `maximum_tilt` | Out-of-plane support for planar distributions. | rad |
| `primary_fraction` | Layered-biaxial fraction near the primary direction. | 0–1 |
| `cross_fraction` | Layered-biaxial fraction near the transverse direction. | 0–1 |
| `maximum_in_plane_deviation` | Biaxial directional scatter. | rad |
| `layer_orientation_seed` | Independent seed for the layer basis. | integer |
| `position` | Center-position distribution. | `uniform`, `layered`, `density_gradient` |
| `position_axis` | Axis used for layers or density gradients. | 0, 1, or 2 |
| `layers` | Number of placement layers. | count |
| `jitter_fraction` | Random layer-position jitter relative to spacing. | fraction |
| `density_exponent` | Shape of a density-gradient distribution. | positive exponent |
| `density_toward_high` | Chooses the high-coordinate side of the gradient. | boolean |
| `minimum_bend_radius` | Optional admissible bend radius for generated fibers. | m or `None` |
| `max_attempts_per_fiber` | Rejection-sampling budget per fiber. | count |
| `material_name` | Material-table name assigned to the population. | string |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
population = tangle.FiberPopulationSettings()
fields = ['count', 'segments_per_fiber', 'seed', 'length_minimum', 'length_maximum', 'nominal_parent_length', 'radius_minimum', 'radius_maximum', 'curvature_amplitude_minimum', 'curvature_amplitude_maximum', 'orientation', 'orientation_axis', 'maximum_angle', 'maximum_tilt', 'primary_fraction', 'cross_fraction', 'maximum_in_plane_deviation', 'layer_orientation_seed', 'position', 'position_axis', 'layers', 'jitter_fraction', 'density_exponent', 'density_toward_high', 'minimum_bend_radius', 'max_attempts_per_fiber', 'material_name']
{name: getattr(population, name) for name in fields}

In [ ]:
# Geometry ranges are sampled independently but reproducibly from seed.
population.count = 100
population.seed = 42
population.length_minimum = 0.2e-3
population.length_maximum = 0.3e-3
population.radius_minimum = 3.5e-6
population.radius_maximum = 9.5e-6
population.curvature_amplitude_minimum = 0.0
population.curvature_amplitude_maximum = 10e-6
population.minimum_bend_radius = 35e-6
# layered_biaxial places 40% near a primary in-plane direction, 40%
# near its transverse direction, and leaves the remaining 20% random.
population.orientation = "layered_biaxial"
# For planar/biaxial distributions, orientation_axis is the plane normal.
population.orientation_axis = [0.0, 0.0, 1.0]
population.primary_fraction = 0.4
population.cross_fraction = 0.4
# Position axis 2 is z, so this creates four through-thickness plies.
population.position = "layered"
population.position_axis = 2
population.layers = 4
generated = tangle.generate_fiber_population(cell, population, name="four plies")
print(len(generated), generated.layers())